# Part 1: Data Preprocessing & Feature Engineering 

## Data Download and Schema Overview

In [1]:
import polars as pl
import pathlib as pathlb
from sklearn.preprocessing import StandardScaler, OneHotEncoder

file_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
base_dir = pathlb.Path('data/raw')
file_name = 'yellow_tripdata_2024-01.parquet'
file_path = base_dir/file_name

## Checks if file downloaded if not it downloads it
if not file_path.is_file() :
    base_dir.mkdir(parents=True, exist_ok=True)
    print(f" Error {file_name} not found downloading...\n")
    pl.read_parquet(file_url).write_parquet(file_path)
taxi_df = pl.read_parquet(file_path)

print(f"{"SCHEMA"}\n")
print(f"{"Name":<25}Type")
for col, type in taxi_df.schema.items() :
    print(f"{col:<25}{type}")


taxi_df.head()

SCHEMA

Name                     Type
VendorID                 Int32
tpep_pickup_datetime     Datetime(time_unit='ns', time_zone=None)
tpep_dropoff_datetime    Datetime(time_unit='ns', time_zone=None)
passenger_count          Int64
trip_distance            Float64
RatecodeID               Int64
store_and_fwd_flag       String
PULocationID             Int32
DOLocationID             Int32
payment_type             Int64
fare_amount              Float64
extra                    Float64
mta_tax                  Float64
tip_amount               Float64
tolls_amount             Float64
improvement_surcharge    Float64
total_amount             Float64
congestion_surcharge     Float64
Airport_fee              Float64


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


### Exploring Lookup Zone Data File

In [2]:
file_name ='lookup.csv'
file_path = base_dir / file_name
file_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

if not file_path.is_file():
    base_dir.mkdir(parents=True, exist_ok=True)
    print(f" Error {file_name} not found downloading...\n")
    pl.read_csv(file_url,encoding="utf8-lossy").write_csv(file_path)

lookup_df = pl.read_csv(file_path)

print(f"{"SCHEMA"}\n")
print(f"{"Name":<25}Type")
for col, type in lookup_df.schema.items() :
    print(f"{col:<25}{type}")

print(lookup_df['Borough'].value_counts())

SCHEMA

Name                     Type
LocationID               Int64
Borough                  String
Zone                     String
service_zone             String
shape: (8, 2)
┌───────────────┬───────┐
│ Borough       ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ Queens        ┆ 69    │
│ EWR           ┆ 1     │
│ Staten Island ┆ 20    │
│ N/A           ┆ 1     │
│ Brooklyn      ┆ 61    │
│ Manhattan     ┆ 69    │
│ Unknown       ┆ 1     │
│ Bronx         ┆ 43    │
└───────────────┴───────┘


## Data Cleaning


### Cleaning Taxi Parquet

In [3]:
cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime',  'PULocationID', 'DOLocationID', 'fare_amount']
initial_rows = len(taxi_df)

taxi_df = taxi_df.drop_nulls()
null_rows = initial_rows - len(taxi_df)

taxi_df = taxi_df.filter( 
    (~pl.col("fare_amount").is_nan()) & 
    (~pl.col("trip_distance").is_nan()))
nan_rows = (initial_rows - null_rows) - len(taxi_df)


taxi_df = taxi_df.filter((pl.col('trip_distance') > 0))
distance_rows = (initial_rows - null_rows - nan_rows) - len(taxi_df)

taxi_df = taxi_df.filter((pl.col('fare_amount') > 0)& (pl.col('fare_amount') <= 500))
fare_rows = (initial_rows - null_rows - nan_rows - distance_rows) - len(taxi_df)

taxi_df = taxi_df.filter(pl.col('tpep_dropoff_datetime') > pl.col('tpep_pickup_datetime'))
time_rows = (initial_rows - null_rows - nan_rows - distance_rows - fare_rows) - len(taxi_df)
## removes incorrect dates in data
taxi_df = taxi_df.filter((pl.col('tpep_pickup_datetime').dt.year() == 2024))

year_rows = (initial_rows - null_rows - nan_rows - distance_rows - fare_rows - time_rows) - len(taxi_df)

removed_rows= initial_rows - len(taxi_df)

print("Data Cleaning Summary\n\n")
print(f"Initial row count: {initial_rows}")
print(f"Final row count: {len(taxi_df)}\n")
print(f"Total rows removed {removed_rows}")
print(f"Rows removed due to null/missing values values: {null_rows}")
print(f"Rows removed due to nan values: {nan_rows}")
print(f"Rows remove due to invalid distance: {distance_rows}")
print(f"Rows remove due to unwanted/invlaid fare range: {fare_rows}")
print(f"Rows remove due to invalid pickup/dropoff times: {time_rows}")
print(f"Rows remove due to dates being out of range : {year_rows}")

taxi_df.write_parquet('data/cleaned_data.parquet')



Data Cleaning Summary


Initial row count: 2964624
Final row count: 2754363

Total rows removed 210261
Rows removed due to null/missing values values: 140162
Rows removed due to nan values: 0
Rows remove due to invalid distance: 37546
Rows remove due to unwanted/invlaid fare range: 32481
Rows remove due to invalid pickup/dropoff times: 58
Rows remove due to dates being out of range : 14


### Cleaning Lookup CSV

In [4]:
lookup_df = lookup_df.filter(
    (pl.col('Borough') != 'Unknown')&
    (pl.col('Borough') != 'N/A'))
print(lookup_df['Borough'].value_counts())

shape: (6, 2)
┌───────────────┬───────┐
│ Borough       ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ Queens        ┆ 69    │
│ EWR           ┆ 1     │
│ Manhattan     ┆ 69    │
│ Bronx         ┆ 43    │
│ Brooklyn      ┆ 61    │
│ Staten Island ┆ 20    │
└───────────────┴───────┘


## Feature Engineering


### Joining Boroughs to Taxi Data Frame

In [5]:
taxi_df = taxi_df.join(
    lookup_df.select(["LocationID", "Borough"]),
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)
taxi_df = taxi_df.join(
    lookup_df.select(["LocationID", "Borough"]),
    left_on="DOLocationID",
    right_on="LocationID",
    how="left",
    suffix="_dropoff" 
)

In [6]:
test_df=taxi_df
#Temporal features
taxi_df = taxi_df.with_columns( 
    #Create col that indexes pickup hour
    (pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour')),
    #Create col that indexes weekday, starting at 0
    (pl.col('tpep_pickup_datetime').dt.weekday()-1).alias('pickup_day_of_week')
    ).with_columns(
        # Creates boolean weekend col
        (pl.col('pickup_day_of_week') >= 5).alias('is_weekend')
    )
#Trip Features
taxi_df  = taxi_df.with_columns(
    (pl.col('tpep_dropoff_datetime')- pl.col('tpep_pickup_datetime'))
    #Convert trip duration minutes distance to interger
    .dt.total_minutes().alias('trip_duration_minutes')
    ).with_columns(
        #accounts for zero denominator error
        pl.when(pl.col('trip_duration_minutes') > 0)
        .then(pl.col('trip_distance')* 60/(pl.col('trip_duration_minutes')))
        .otherwise(0) #if duration 0 -> just 0 mph
        .alias('trip_speed_mph')
    )
taxi_df = taxi_df.with_columns(
    pl.col('trip_distance').log1p().alias('log_trip_distance'))
#Fare Features
taxi_df = taxi_df.with_columns(
        pl.when(pl.col('trip_distance') > 0)
        .then(pl.col('fare_amount')/(pl.col('trip_distance')))
        .otherwise(0) #if distance 0 -> just 0
        .alias('fare_per_mile'),
        pl.when(pl.col('trip_duration_minutes') > 0)
        .then(pl.col('fare_amount')/(pl.col('trip_duration_minutes')))
        .otherwise(0) #if distance 0 -> just 0
        .alias('fare_per_minute')
        )

In [7]:
#Zone Features

# 1. Initialize Encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# 2. Select the columns exactly as they appear
# Note: Use the actual names from your df.columns if they differ
cat_features = taxi_df.select(['Borough', 'Borough_dropoff']).to_numpy()

# 3. Fit and Transform
encoded_array = encoder.fit_transform(cat_features)

# 4. Get feature names for the summary report (Requirement 3d)
encoded_cols = encoder.get_feature_names_out(['Borough', 'Borough_dropoff'])

# 5. Add back to dataframe
encoded_df = pl.from_numpy(encoded_array, schema=list(encoded_cols))

# Find columns that exist in both DataFrames to avoid DuplicateError
common_cols = set(taxi_df.columns).intersection(set(encoded_df.columns))

# Also drop the original string columns
cols_to_drop = list(common_cols) + ['Borough', 'Borough_dropoff']
# Use list(set()) to ensure we don't try to drop the same name twice
taxi_df = taxi_df.drop([c for c in set(cols_to_drop) if c in taxi_df.columns])

# Now concatenate safely
taxi_df = pl.concat([taxi_df, encoded_df], how="horizontal")

In [8]:
# Select only the columns that start with 'Borough_' (the pickups)
pickup_dist = taxi_df.select([
    pl.col("^Borough_.*$").sum()
])

print("Pickup Distribution:")
print(pickup_dist)

Pickup Distribution:
shape: (1, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Borough_B ┆ Borough_B ┆ Borough_E ┆ Borough_M ┆ … ┆ Borough_d ┆ Borough_d ┆ Borough_d ┆ Borough_ │
│ ronx      ┆ rooklyn   ┆ WR        ┆ anhattan  ┆   ┆ ropoff_Ma ┆ ropoff_Qu ┆ ropoff_St ┆ dropoff_ │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ nhattan   ┆ eens      ┆ aten      ┆ None     │
│ f64       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ Island    ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ f64       ┆ f64       ┆ ---       ┆ f64      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 4938.0    ┆ 16752.0   ┆ 49.0      ┆ 2.469279e ┆ … ┆ 2.483672e ┆ 125380.0  ┆ 596.0     ┆ 24476.0  │
│           ┆           ┆           ┆ 6         ┆   ┆ 6

### Target Variable Encoding

In [9]:
taxi_df = taxi_df.with_columns(
    (pl.col("tip_amount") > (pl.col("fare_amount") * 0.20))
    .cast(pl.Int8)
    .alias("high_tip")
)
print(taxi_df["high_tip"].value_counts())
taxi_df = taxi_df.drop_nulls(subset=cols + ["tip_amount", "high_tip"])

shape: (2, 2)
┌──────────┬─────────┐
│ high_tip ┆ count   │
│ ---      ┆ ---     │
│ i8       ┆ u32     │
╞══════════╪═════════╡
│ 1        ┆ 1745223 │
│ 0        ┆ 1009140 │
└──────────┴─────────┘


### Data Splitting and Scaling


In [11]:
from sklearn.model_selection import train_test_split
import polars as pl

# 1. First, ensure the data is 100% clean to avoid that NaN error again
# We drop rows where either the features (cols) or targets are missing
taxi_df = taxi_df.drop_nulls(subset=cols + ["tip_amount", "high_tip"])

# 2. Convert to Pandas for the stratified split
df_pandas = taxi_df.to_pandas()

# 3. Split 1: Separate Training (70%) from the rest (30%)
train_raw, temp_raw = train_test_split(
    df_pandas, 
    test_size=0.30, 
    random_state=42, 
    stratify=df_pandas["high_tip"]
)

# 4. Split 2: Divide the 30% into Validation (15%) and Test (15%)
val_raw, test_raw = train_test_split(
    temp_raw, 
    test_size=0.50, 
    random_state=42, 
    stratify=temp_raw["high_tip"]
)

# 5. Convert back to Polars for memory efficiency
train_df = pl.from_pandas(train_raw)
val_df = pl.from_pandas(val_raw)
test_df = pl.from_pandas(test_raw)

print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")
for name, df in {"Train": train_df, "Val": val_df, "Test": test_df}.items():
    percentage = df["high_tip"].mean() * 100
    print(f"{name} High Tip Ratio: {percentage:.2f}%")

Train size: 1928054 | Val size: 413154 | Test size: 413155
Train High Tip Ratio: 63.36%
Val High Tip Ratio: 63.36%
Test High Tip Ratio: 63.36%


___

In [12]:
from sklearn.preprocessing import StandardScaler

# 1. Initialize the Scaler
scaler = StandardScaler()

# 2. Fit ONLY on the training data (to prevent data leakage)
scaler.fit(train_df[cols].to_numpy())

# 3. Transform all three sets
# We convert to numpy because scikit-learn performs faster on arrays than Polars/Pandas frames
X_train = scaler.transform(train_df[cols].to_numpy())
X_val = scaler.transform(val_df[cols].to_numpy())
X_test = scaler.transform(test_df[cols].to_numpy())

# 4. Grab your targets (no scaling needed for targets in classification)
y_train_reg = train_df["tip_amount"].to_numpy()
y_train_clf = train_df["high_tip"].to_numpy()

y_val_reg = val_df["tip_amount"].to_numpy()
y_val_clf = val_df["high_tip"].to_numpy()

print("Scaling complete. Data is now centered at 0 with a unit variance.")

Scaling complete. Data is now centered at 0 with a unit variance.


In [13]:
import pandas as pd

split_stats = []
for name, df in {"Train": train_df, "Val": val_df, "Test": test_df}.items():
    count = len(df)
    high_tip_pct = (df["high_tip"].mean() * 100)
    split_stats.append({"Split": name, "Samples": f"{count:,}", "High Tip %": f"{high_tip_pct:.2f}%"})

summary_df = pd.DataFrame(split_stats)
print("### Data Split Summary")
print(summary_df.to_string(index=False))

### Data Split Summary
Split   Samples High Tip %
Train 1,928,054     63.36%
  Val   413,154     63.36%
 Test   413,155     63.36%


___

In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 1. Linear Regression
# Since data is already scaled, we don't need a pipeline here
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train_reg)

# 2. Random Forest Regressor
# Using 100 trees, but keeping depth at 10 to manage memory/thermals
rf_reg = RandomForestRegressor(
    n_estimators=100, 
    max_depth=10, 
    n_jobs=-1, 
    random_state=42
)
rf_reg.fit(X_train, y_train_reg)

# 3. Evaluation on Validation Set
models = {"Linear Regression": lin_reg, "Random Forest": rf_reg}

print(f"{'Model':<20} | {'MAE':<10} | {'RMSE':<10} | {'R2 Score':<10}")
print("-" * 55)

for name, model in models.items():
    preds = model.predict(X_val)
    mae = mean_absolute_error(y_val_reg, preds)
    rmse = np.sqrt(mean_squared_error(y_val_reg, preds))
    r2 = r2_score(y_val_reg, preds)
    print(f"{name:<20} | {mae:<10.4f} | {rmse:<10.4f} | {r2:<10.4f}")

Model                | MAE        | RMSE       | R2 Score  
-------------------------------------------------------
Linear Regression    | 1.9078     | 3.0295     | 0.3808    
Random Forest        | 1.8225     | 2.8981     | 0.4333    


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 1. Logistic Regression
# We use the 'sag' solver because it's optimized for large datasets (2M+ rows)
log_clf = LogisticRegression(max_iter=1000, solver='sag', random_state=42)
log_clf.fit(X_train, y_train_clf)

# 2. Random Forest Classifier
# 100 trees, max_depth 10 to keep your M2 Air/Colab from hitting a memory wall
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
rf_clf.fit(X_train, y_train_clf)

# 3. Evaluation
models_clf = {"Logistic Regression": log_clf, "Random Forest": rf_clf}

print(f"{'Model':<20} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10}")
print("-" * 55)

for name, model in models_clf.items():
    preds = model.predict(X_val)
    acc = accuracy_score(y_val_clf, preds)
    prec = precision_score(y_val_clf, preds)
    rec = recall_score(y_val_clf, preds)
    print(f"{name:<20} | {acc:<10.4f} | {prec:<10.4f} | {rec:<10.4f}")

Model                | Accuracy   | Precision  | Recall    
-------------------------------------------------------
Logistic Regression  | 0.6339     | 0.6365     | 0.9842    
Random Forest        | 0.6480     | 0.6484     | 0.9709    


In [16]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# 1. Define the parameter grid
# We stay within reasonable bounds to avoid RAM "thrashing"
param_distributions = {
    'n_estimators': [50, 100, 150],
    'max_depth': [10, 20, 30, None],
    'min_samples_leaf': [1, 2, 4]
}

# 2. Setup Randomized Search
# n_iter=5 means it tries 5 random combinations
# cv=3 means 3-fold cross-validation
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
    param_distributions=param_distributions,
    n_iter=5, 
    cv=3, 
    verbose=2, 
    random_state=42,
    scoring='accuracy'
)

# 3. Fit on a 100k subset for efficiency
print("Tuning on 100,000 samples...")
random_search.fit(X_train[:100000], y_train_clf[:100000])

# 4. Results
print(f"Best Parameters: {random_search.best_params_}")
best_model = random_search.best_estimator_

# 5. Final validation with best model
tuned_preds = best_model.predict(X_val)
print(f"Tuned Model Accuracy: {accuracy_score(y_val_clf, tuned_preds):.4f}")

Tuning on 100,000 samples...
Fitting 3 folds for each of 5 candidates, totalling 15 fits
[CV] END max_depth=None, min_samples_leaf=4, n_estimators=150; total time=   6.1s
[CV] END max_depth=None, min_samples_leaf=4, n_estimators=150; total time=   6.1s
[CV] END max_depth=None, min_samples_leaf=4, n_estimators=150; total time=   6.1s
[CV] END .max_depth=20, min_samples_leaf=2, n_estimators=100; total time=   3.6s
[CV] END .max_depth=20, min_samples_leaf=2, n_estimators=100; total time=   3.6s
[CV] END .max_depth=20, min_samples_leaf=2, n_estimators=100; total time=   3.6s
[CV] END .max_depth=30, min_samples_leaf=4, n_estimators=150; total time=   6.0s
[CV] END .max_depth=30, min_samples_leaf=4, n_estimators=150; total time=   6.0s
[CV] END .max_depth=30, min_samples_leaf=4, n_estimators=150; total time=   6.0s
[CV] END max_depth=None, min_samples_leaf=2, n_estimators=50; total time=   2.2s
[CV] END max_depth=None, min_samples_leaf=2, n_estimators=50; total time=   2.3s
[CV] END max_dept

In [17]:
from sklearn.model_selection import cross_val_score

# 1. Prepare a stratified sample of 200,000 rows for speed
# Using the first 200k since the original split was already stratified
X_sample = X_train[:200000]
y_sample = y_train_clf[:200000]

# 2. Initialize the model with your "Best Parameters"
cv_model = RandomForestClassifier(
    n_estimators=100, 
    min_samples_leaf=4, 
    max_depth=20, 
    n_jobs=-1, 
    random_state=42
)

# 3. Perform 5-Fold CV
print("Performing 5-Fold Cross-Validation on 200k rows...")
cv_scores = cross_val_score(cv_model, X_sample, y_sample, cv=5, scoring='accuracy')

# 4. Report results
print(f"Scores for each fold: {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

Performing 5-Fold Cross-Validation on 200k rows...
Scores for each fold: [0.64505  0.6449   0.64285  0.643175 0.644225]
Mean CV Accuracy: 0.6440
Standard Deviation: 0.0009


____


In [20]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Architecture Design
# We use 'relu' for hidden layers and 'sigmoid' for the final binary output
nn_model = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)), # Input layer matching our features
    
    # Hidden Layer 1: 64 neurons
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2), # Regularization: prevents overfitting
    
    # Hidden Layer 2: 32 neurons
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.1),
    
    # Output Layer: 1 neuron with Sigmoid (Probability 0 to 1)
    layers.Dense(1, activation='sigmoid')
])

# 2. Compilation
# Binary Crossentropy is the standard loss function for classification
nn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 3. Training
# Large batch_size is crucial for OS efficiency when dealing with millions of rows
history = nn_model.fit(
    X_train, y_train_clf,
    validation_data=(X_val, y_val_clf),
    epochs=20,
    batch_size=2048, 
    verbose=1
)

# 4. Final Evaluation
nn_val_acc = nn_model.evaluate(X_val, y_val_clf, verbose=0)[1]
print(f"\nNeural Network Validation Accuracy: {nn_val_acc:.4f}")

Epoch 1/20
942/942 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6337 - loss: 0.6532 - val_accuracy: 0.6347 - val_loss: 0.6514
Epoch 2/20
942/942 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6346 - loss: 0.6516 - val_accuracy: 0.6348 - val_loss: 0.6512
Epoch 3/20
942/942 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6347 - loss: 0.6514 - val_accuracy: 0.6348 - val_loss: 0.6510
Epoch 4/20
942/942 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6348 - loss: 0.6511 - val_accuracy: 0.6352 - val_loss: 0.6509
Epoch 5/20
942/942 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6350 - loss: 0.6509 - val_accuracy: 0.6349 - val_loss: 0.6506
Epoch 6/20
942/942 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6349 - loss: 0.6507 - val_accuracy: 0.6352 - val_loss: 0.6503
Epoch 7/20
942/942 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6350 - loss: 0.6505 - val_accuracy: 0.6353 - val_loss: 0.6502
Epoch 8/20
942/942 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6351 - loss: 0.6503 - val_accuracy: 0.

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. Prepare PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train_clf, dtype=torch.float32).unsqueeze(1)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val_clf, dtype=torch.float32).unsqueeze(1)

# 2. Create DataLoader (Batch Processing)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=2048, shuffle=True)

# 3. Define the Architecture
class TaxiNN(nn.Module):
    def __init__(self, input_dim):
        super(TaxiNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1) # No Sigmoid here because we use BCEWithLogitsLoss
        )
    def forward(self, x):
        return self.net(x)

model = TaxiNN(X_train.shape[1])
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Training Loop (20 Epochs)
print("Starting 20 Epochs of Training...")
for epoch in range(20):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/20], Loss: {total_loss/len(train_loader):.4f}")

# 5. Final Accuracy
model.eval()
with torch.no_grad():
    y_pred_logits = model(X_val_t)
    y_pred = (torch.sigmoid(y_pred_logits) > 0.5).float()
    accuracy = (y_pred == y_val_t).float().mean()
    print(f"\nFinal Neural Network Accuracy: {accuracy:.4f}")

Starting 20 Epochs of Training...
Epoch [5/20], Loss: 0.6505
Epoch [10/20], Loss: 0.6495
Epoch [15/20], Loss: 0.6481
Epoch [20/20], Loss: 0.6473

Final Neural Network Accuracy: 0.6406


In [ ]:
import matplotlib.pyplot as plt

def plot_loss(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history['train_loss'], label='Training Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Neural Network Loss Curves (20 Epochs)')
    plt.xlabel('Epochs')
    plt.ylabel('BCE Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

# Assuming your training loop stored losses in a dictionary:
# plot_loss(history_dict)